# 02 Preprocessing and Client Splits

Loads each dataset, preprocesses features, creates binary labels, and writes non-IID client splits.

In [ ]:
from pathlib import Path
import os
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError(
        'Cannot find EcoTrust-FLIDS IMPLEMENTATION. In Colab, mount Drive and set ECOTRUST_PROJECT_ROOT to '
        '/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION.'
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
else:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'
RUN_MODE = 'full'  # full run over each dataset's canonical configured source
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)

import importlib
import ecotrust.data.loaders as loaders
import ecotrust.data.preprocess as preprocess
import ecotrust.data.clients as clients
importlib.reload(loaders)
importlib.reload(preprocess)
importlib.reload(clients)

from ecotrust.config import load_config, run_mode_settings
from ecotrust.data.loaders import load_dataset_frame
from ecotrust.data.preprocess import preprocess_frame
from ecotrust.data.clients import dirichlet_client_splits, save_splits
from ecotrust.io import write_json
cfg = load_config(CONFIG_PATH)
settings = run_mode_settings(cfg, RUN_MODE)
REQUIRED_OUTPUTS = [
    'X_train.parquet', 'X_test.parquet', 'y_train.parquet', 'y_test.parquet',
    'preprocessor.joblib', 'client_splits.json', 'preprocess_manifest.json'
]
COLAB_FULL_ROW_CAPS = {
    # CICIoT2023 train.csv is about 1.62 GB uncompressed. Reading it fully from Drive can crash Colab RAM.
    'CICIoT2023': 250_000,
}

for dataset in DATASETS:
    out = cfg.results_root / dataset / 'preprocessed' / RUN_MODE
    complete = all((out / name).exists() and (out / name).stat().st_size > 0 for name in REQUIRED_OUTPUTS)
    if complete:
        print('Skipping completed', dataset, '->', out)
        continue

    print('Preprocessing', dataset)
    nrows = settings['rows_per_dataset']
    if RUN_MODE == 'full' and str(PROJECT_ROOT).startswith('/content/drive'):
        nrows = COLAB_FULL_ROW_CAPS.get(dataset, nrows)
        if nrows is not None:
            print(' Using Colab-safe row cap:', nrows)

    # Use one canonical configured source in both smoke and full modes. Loading every alternate archive duplicates data and can exhaust Colab RAM.
    frame = load_dataset_frame(dataset, cfg, nrows=nrows, max_sources=1)
    result = preprocess_frame(frame, cfg.datasets[dataset], out, random_state=cfg.seed)
    splits = dirichlet_client_splits(result.y_train, n_clients=settings['max_clients'], seed=cfg.seed)
    save_splits(splits, out / 'client_splits.json')
    write_json(out / 'preprocess_manifest.json', {
        'dataset': dataset, 'mode': RUN_MODE, 'rows_loaded': len(frame),
        'row_cap': nrows,
        'n_features': len(result.feature_names), 'n_clients': len(splits),
        'label_column': result.label_column, 'attack_type_column': result.attack_type_column,
    })
    print(' wrote', out)
